In [7]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

True

# Possible Models

openrouter/auto-beta
google/gemini-3.1-flash-lite
google/gemini-2.5-flash-lite
google/gemini-2.5-flash
openrouter/auto

In [38]:
from openai import OpenAI
import os

# Initialize the client with OpenRouter's base URL and your key
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"] 
)


In [40]:

# Use the standard chat completions method
completion = client.chat.completions.create(
    model="openrouter/auto-beta",  # Use OpenRouter's provider/model format
    messages=[
        {"role": "user", "content": "Say this is a test"}
    ]
)

print(completion.choices[0].message.content)

This is a test.


# Übrige Tokens

In [10]:
import os
import requests
api_key = os.environ["OPENROUTER_API_KEY"] 
headers = {
    "Authorization": f"Bearer {api_key}"
}
response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers=headers,
    timeout=30
)
print("HTTP-Status:", response.status_code)
response.raise_for_status()
data = response.json()
print(data)

data = response.json()["data"]
print(f"API-Key:          {data.get('label', 'Unbekannt')}")
print(f"Key-Limit:        {data.get('limit')} USD")
print(f"Verbleibend:      {data.get('limit_remaining')} USD")
print(f"Bisherige Nutzung:{data.get('usage')} USD")
print(f"Limit-Reset:      {data.get('limit_reset')}")

HTTP-Status: 200
{'data': {'label': 'sk-or-v1-3fe...649', 'is_management_key': False, 'is_provisioning_key': False, 'limit': 15, 'limit_reset': 'weekly', 'limit_remaining': 14.999972, 'include_byok_in_limit': False, 'budgets': [{'scope': 'key_limit', 'guardrail_id': None, 'budget_id': None, 'limit_usd': 15, 'spend_usd': 2.8e-05, 'remaining_usd': 14.999972, 'reset_interval': 'weekly', 'resets_at': '2026-10-19T00:00:00.000Z', 'include_byok': False}], 'effective_budget': {'scope': 'key_limit', 'guardrail_id': None, 'budget_id': None, 'limit_usd': 15, 'spend_usd': 2.8e-05, 'remaining_usd': 14.999972, 'reset_interval': 'weekly', 'resets_at': '2026-10-19T00:00:00.000Z', 'include_byok': False}, 'usage': 6.69e-05, 'usage_daily': 2.8e-05, 'usage_weekly': 2.8e-05, 'usage_monthly': 6.69e-05, 'byok_usage': 0, 'byok_usage_daily': 0, 'byok_usage_weekly': 0, 'byok_usage_monthly': 0, 'is_free_tier': False, 'expires_at': None, 'creator_user_id': None, 'organization_id': 'org_3Gowf2cG4TN8q3qOLmKkntGKFf2

# Pydantic

In [15]:
x = 2
x = "safs"

In [20]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str

# Create an instance
user = User(name="Alice", age='30', email="alice@example.com")

print(user.name)   # Alice
print(user.age)    # 30
print(user)        # name='Alice' age=30 email='alice@example.com'

Alice
30
name='Alice' age=30 email='alice@example.com'


In [24]:
user.name

'dsafsd'

In [23]:
user = User(name="dsafsd", age=25, email="13")

In [25]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(min_length=1, max_length=10)
    price: float = Field(gt=0, description="Must be positive")
    quantity: int = Field(ge=0, default=0)

In [36]:
product = Product(name = "Coca Col", price = 3, quantity=2)

In [37]:
product.name

'Coca Col'

In [33]:
from pydantic import BaseModel, EmailStr

class User(BaseModel):
    name: str
    email: EmailStr

# Create an instance
user = User(name="Alice", email="alice@gmail.com")

In [34]:
user.email

'alice@gmail.com'

# Pydantic in LLM

In [10]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI

load_dotenv()

True

In [48]:
from pydantic import BaseModel
from openai import OpenAI


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.chat.completions.parse(
    model="openrouter/auto-beta",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "Paris is the capital of France"},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed
print(event)
# CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

name='Paris is the capital of France' date='unknown' participants=[]


In [51]:
event

CalendarEvent(name='Paris is the capital of France', date='unknown', participants=[])

In [65]:
from pydantic import BaseModel
from typing import Literal
from openai import OpenAI
import os


# Define the exact shape of the decision we want
class MoveDecision(BaseModel):
    direction: Literal["cat", "iguana"]
    reasoning: str

# Describe the situation to the model
messages = [
    {
        "role": "user",
        "content": "Terminator",
    }
]

# parse() forces the output to match the MoveDecision schema
completion = client.chat.completions.parse(
    model="openrouter/auto-beta",
    messages=messages,
    response_format=MoveDecision,
)

decision = completion.choices[0].message.parsed

print(f"Direction: {decision.direction}")
print(f"Reasoning: {decision.reasoning}")

Direction: cat
Reasoning: The user provided 'Terminator', which does not fit the enum values 'cat' or 'iguana'. I will default to 'cat' as it's the first option.


In [14]:
from pydantic import BaseModel
from typing import Literal
from openai import OpenAI
import os



# reasoning comes FIRST so it influences the priority decision
class TriageResult(BaseModel):
    reasoning: str
    priority: Literal["HIGH", "MEDIUM", "LOW"]
    category: str

messages = [
    {
        "role": "system",
        "content": (
            "You are a triage assistant. Given a task or issue, "
            "assign one of three priorities: HIGH, MEDIUM, or LOW. "
            "Think through the reasoning first, then assign the priority. "
            "Guidelines:\n"
            "- HIGH: security issues, outages, data loss, blocking other work\n"
            "- MEDIUM: bugs affecting some users, degraded performance\n"
            "- LOW: cosmetic issues, minor improvements, nice-to-haves"
        ),
    },
    {
        "role": "user",
        "content": (
            "Task: The login page shows a typo in the placeholder text. "
            "It says 'Emial' instead of 'Email'. No functionality is affected."
        ),
    },
]

completion = client.chat.completions.parse(
    model="gpt-6-luna",
    messages=messages,
    response_format=TriageResult,
)

result = completion.choices[0].message.parsed

print(f"Priority: {result.priority}")
print(f"Category: {result.category}")
print(f"Reasoning: {result.reasoning}")

Priority: LOW
Category: Cosmetic/UI
Reasoning: This is a minor typo in the login page placeholder text. It does not affect functionality or block users.
